# Featune: from data to reviewable feature search

This notebook runs one classification experiment on the bundled Adult Income sample. Follow the cells in order: **data and untouched holdout → Schema → baseline and search → trial evidence → holdout score → export → optional LLM**.

Install from the repository root with `pip install -e '.[dev]'`. The default local model is pinned TabPFN v2; the first fit downloads its weights. Steps 1–6 need no LLM credentials. Field descriptions cannot by themselves prove that a column was available at the prediction time.

## 1 · Load data and reserve the test set first

`X` must be a DataFrame with unique column names; `y` is a one-dimensional target. Stratify the split, then pass only `train_x/train_y` to `study.optimize`. Keep `test_x/test_y` untouched until search is complete. The file lookup works when Jupyter starts from the repository root or `examples/`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import featune

# Works when Jupyter starts in the repository root or in examples/.
csv_path = next(
    (p for p in [Path("examples/data/adult_income.csv"), Path("data/adult_income.csv")] if p.exists()), None
)
if csv_path is None:
    raise FileNotFoundError("Start Jupyter from the repository root or examples/ directory")
repo_root = csv_path.resolve().parents[2]
data = pd.read_csv(csv_path).sample(n=400, random_state=42)
X = data.drop(columns="income_over_50k")
y = data["income_over_50k"]
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
print(f"Rows: {len(data)} | search rows: {len(train_x)} | untouched test rows: {len(test_x)}")
print("Target distribution:", y.value_counts().to_dict())
display(train_x.head(3))

## 2 · Describe fields and the prediction task

`DatasetSchema` tells the search about usable columns, target meaning and prediction time. Each `FieldSchema` supplies a name, type and business description. All input columns here are numeric; `encoding` explains the 0/1 indicators. The target was removed from `X`. `available_at` is descriptive metadata, not proof of availability.

In [ ]:
ENCODING = "1=yes, 0=no"
AVAILABLE_AT = "at survey time"
OBJECTIVE = "Estimate whether annual income exceeds 50K from survey-time fields"
TARGET_DEFINITION = "1 means annual income > 50K; 0 means <= 50K"
PREDICTION_POINT = "At survey collection, after the listed fields are recorded"

descriptions = {
    "age": "Age in years",
    "fnlwgt": "Census sample weight",
    "education_num": "Numeric education code; larger values mean more education",
    "capital_gain": "Reported annual capital gain",
    "capital_loss": "Reported annual capital loss",
    "hours_per_week": "Usual working hours per week",
    "workclass_Private": "Works for a private employer",
    "education_Bachelors": "Highest education is a bachelor degree",
    "education_Masters": "Highest education is a master degree",
    "marital_status_Married_civ_spouse": "Married with a civilian spouse",
    "occupation_Exec_managerial": "Works in an executive or managerial role",
    "occupation_Prof_specialty": "Works in a professional specialty",
    "relationship_Husband": "Household relationship is husband",
    "sex_Male": "Sex is recorded as male",
    "native_country_United_States": "Native country is the United States",
    "race_White": "Race is recorded as White",
}
schema = featune.DatasetSchema(
    fields=[
        featune.FieldSchema(
            name=name,
            dtype="numeric",
            description=description,
            encoding=ENCODING
            if name
            not in {"age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"}
            else None,
            available_at=AVAILABLE_AT,
        )
        for name, description in descriptions.items()
    ],
    objective=OBJECTIVE,
    target_definition=TARGET_DEFINITION,
    prediction_point=PREDICTION_POINT,
)
schema.validate_frame(train_x)
print(f"Usable fields: {len(schema.usable)} | target excluded: {'income_over_50k' not in schema.usable}")

### Preview one controlled feature

`FeatureSpec` accepts only allowlisted operators (here `divide`), existing input columns and valid parameters; `Hypothesis` stores a rationale to test. `FeatureCompiler.fit(train_x)` learns any stateful transformation on training rows only, and `transform` reuses it. This cell merely previews computed training values; it does **not** force the feature into the selected model. Every search candidate still goes through validation and CV.

In [ ]:
preview_spec = featune.FeatureSpec(
    name="hours_per_age",
    op="divide",
    inputs=["hours_per_week", "age"],
    hypothesis=featune.Hypothesis(
        concept="work intensity",
        rationale="Working hours relative to age may carry predictive information",
        expected_direction="unknown",
    ),
)
compiler = featune.FeatureCompiler(schema, [preview_spec]).fit(train_x)
display(compiler.transform(train_x).head(3)[["hours_per_week", "age", "hours_per_age"]])

## 3 · Evaluate the baseline, then try two candidates

`create_study` configures the primary metric, sampler, search policy and limits. `RandomSampler` only **proposes** closed-DSL expressions; `CVEvaluator` compiles and scores every candidate on the same training folds. `n_trials=2` allows two candidate attempts in addition to baseline trial 0. `max_model_fits=7` covers two baseline fits, at most four candidate fits and the final refit. The baseline can remain the winner.

In [ ]:
study = featune.create_study(
    metric="auc",  # higher is better; baseline remains eligible
    sampler=featune.RandomSampler(seed=42),
    search_strategy="greedy",  # extend the best completed feature set
    max_features=2,  # at most two derived columns
    budget=featune.SearchBudget(max_model_fits=7),
)
evaluator = featune.CVEvaluator(cv=2, importance_repeats=0)
study.optimize(train_x, train_y, schema, evaluator=evaluator, n_trials=2)
print(f"Baseline inner-CV AUC: {study.baseline_trial.value:.4f}")
print(f"Best inner-CV AUC:     {study.best_value:.4f}")
print(f"Best trial: {study.best_trial.number} | derived columns: {len(study.best_features)}")

## 4 · Read the trials, not just the highest score

`value` is inner-CV AUC. `delta` is candidate minus baseline; `parent_delta` is candidate minus its parent. Higher AUC is better, so positive differences are improvements. `feature_history()` reports operators, inputs and hypothesis status. With permutation importance disabled, some evidence is `inconclusive`; these labels are not causal findings. `search_summary()` reports actual resource use.

In [ ]:
trials = study.trials_dataframe()
display(
    trials[["number", "parent", "state", "value", "delta", "parent_delta", "feature_count", "model_fits"]]
)
display(
    study.feature_history()[["trial", "feature", "op", "inputs", "status", "reason"]]
    if study.best_features
    else study.feature_history()
)
print("Resource summary:", study.search_summary())

## 5 · Score the untouched test set once

`export_pipeline()` returns the best feature compiler and model, refitted on all **search-training rows**. The holdout AUC is a separate estimate. Repeatedly tuning against this score would turn the holdout into development data. The inner-CV and holdout scores use different rows; their difference is not a feature contribution.

In [ ]:
pipeline = study.export_pipeline()  # best feature compiler + fitted model
test_probability = pipeline.predict_proba(test_x)[:, 1]
test_auc = roc_auc_score(test_y, test_probability)
print(f"Untouched test AUC: {test_auc:.4f}")
display(pd.DataFrame({"actual": test_y.to_numpy()[:5], "probability_of_1": test_probability[:5]}))

## 6 · Save reviewable results

`features.json` contains the selected controlled DSL, `pipeline.joblib` is ready for `predict/predict_proba`, and `report.html` shows the interactive trial trajectory. Generated artifacts live under Git-ignored `runs/`. **The joblib model may contain training context: load only trusted files and protect it as training data.**

In [ ]:
output = repo_root / "runs" / "notebook-en"
output.mkdir(parents=True, exist_ok=True)
study.export_features(output / "features.json")
study.export_pipeline(output / "pipeline.joblib")
study.report(output / "report.html")
print("Saved reviewable results to:", output)

## 7 · Optional: let an LLM propose one feature

Load `FEATUNE_API_KEY`, `FEATUNE_BASE_URL` and `FEATUNE_MODEL` from the project `.env` into this Jupyter process first, for example with `set -a; source .env; set +a; jupyter notebook` from the repository root. `FEATUNE_PROVIDER=anthropic|openai|local` is optional; the default is Anthropic Messages. Set `RUN_LLM=True` in the next cell to send a request; it skips by default to avoid unexpected charges. The LLM proposes JSON only: Featune still validates the DSL and evaluates it with the same CV path. This run allows one proposal, one column and 30,000 known tokens.

Protocol map: `anthropic` uses `/v1/messages`; `openai` uses `/chat/completions` with `max_completion_tokens`; `local` uses the OpenAI-compatible shape with `max_tokens`. A legacy gateway can set `LLMSampler(..., output_token_parameter="max_tokens")`. Temperature is omitted unless explicitly set.

In [ ]:
import os

RUN_LLM = False  # Change to True only when you want to send a billable request.
required = ("FEATUNE_API_KEY", "FEATUNE_BASE_URL", "FEATUNE_MODEL")
missing = [name for name in required if not os.getenv(name)]
if not RUN_LLM:
    print("LLM run skipped; set RUN_LLM=True to opt in.")
elif missing:
    print("LLM run skipped; set these environment variables:", ", ".join(missing))
else:
    llm_sampler = featune.LLMSampler(
        provider=os.getenv("FEATUNE_PROVIDER", "anthropic"),
        model=os.environ["FEATUNE_MODEL"],
        base_url=os.environ["FEATUNE_BASE_URL"],
        max_tokens=1024,
        features_per_trial=1,
    )
    llm_study = featune.create_study(
        metric="auc",
        sampler=llm_sampler,
        search_strategy="greedy",
        max_features=1,
        budget=featune.SearchBudget(max_model_fits=4, max_llm_tokens=30_000),
    )
    llm_study.optimize(
        train_x,
        train_y,
        schema,
        evaluator=featune.CVEvaluator(cv=2),
        n_trials=1,
        refit=False,
    )
    display(llm_study.proposal_history())
    print("LLM trial state:", llm_study.trials[-1].state)
    print("Best inner-CV AUC:", round(llm_study.best_value, 4))

## Next steps

- Replace the DataFrame, target and field descriptions in steps 1–2; verify business-time availability.
- `search_strategy="autonomous"` adds or removes one feature at a time; `max_features` is a ceiling.
- For regression, use `metric="rmse"`; the default switches to `TabPFNRegressor`, and a negative `delta` improves the loss.
- See the [guide](../docs/en/guide.md) and [API reference](../docs/en/reference.md) for grouped/time CV, budgets and resume.